# Pipeline Validation: RF + GB Ensemble → Erlang-A Optimizer

This notebook proves the core math works end-to-end:
1. **Demand Forecast** — Load the pre-trained RF + GB ensemble, predict call volume for today's 24 business slots
2. **Erlang-A Optimizer** — Feed predicted demand into the queueing-theory optimizer to compute minimum staffing

No Docker, no FastAPI, no React — just pure ML + math.

In [2]:
# =============================================================================
# Cell 1: Setup and Imports
# =============================================================================
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar

# Add project root so we can import our custom modules
PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> project root
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Import directly from submodules to avoid triggering HybridForecaster's xgboost import
from main_module.workforce.call_center_emulator import CallCenterEmulator, EmulatorConfig
from main_module.workforce.supply_optimizer import SupplyOptimizer, OptimizationConstraints

MODEL_PATH = PROJECT_ROOT / "data" / "models" / "call_volume_model_bundle.pkl"
print(f"Project root : {PROJECT_ROOT}")
print(f"Model path   : {MODEL_PATH}")
print(f"Model exists : {MODEL_PATH.exists()}")

Project root : /Users/jackiewang/Documents/GitHub/DSC180B-Q2-Project
Model path   : /Users/jackiewang/Documents/GitHub/DSC180B-Q2-Project/data/models/call_volume_model_bundle.pkl
Model exists : True


In [3]:
# =============================================================================
# Cell 2: The Demand Forecast (RF + GB Ensemble)
# =============================================================================

# --- 2a. Load the pre-trained model bundle ---
with open(MODEL_PATH, "rb") as f:
    bundle = pickle.load(f)

rf_model          = bundle["rf_model"]
gb_model          = bundle["gb_model"]
features          = bundle["features"]
forecast_weeks    = bundle["forecast_weeks"]
holiday_profiles  = bundle["holiday_profiles"]
call_volume_hist  = bundle["call_volume_history"]

print(f"Trained at       : {bundle['trained_at']}")
print(f"Forecast weeks   : {forecast_weeks}")
print(f"Features ({len(features)}):")
for f_name in features:
    print(f"  - {f_name}")
print(f"History range    : {call_volume_hist.index.min()} → {call_volume_hist.index.max()}")
print(f"History rows     : {len(call_volume_hist):,}")

Trained at       : 2026-03-06T21:03:43.010965
Forecast weeks   : 1
Features (15):
  - hour_sin
  - hour_cos
  - day_sin
  - day_cos
  - month
  - weekofyear
  - is_january
  - is_minor_holiday
  - is_major_holiday
  - days_to_tax_day
  - is_tax_season
  - is_post_tax_drop
  - lag_1weeks
  - trend_1w
  - max_1w
History range    : 2023-11-28 00:00:00 → 2025-11-15 23:30:00
History rows     : 34,512


In [4]:
# --- 2b. Build today's 24 business slots (5:00 AM – 4:30 PM, every 30 min) ---
today = pd.Timestamp.now().normalize()
print(f"Forecasting for: {today.strftime('%A, %B %d, %Y')}")

slots = pd.date_range(
    start=today + pd.Timedelta(hours=5),
    end=today + pd.Timedelta(hours=16, minutes=30),
    freq="30min",
)

df = pd.DataFrame(index=slots)
df["Call_Volume"] = np.nan  # future — unknown

# --- 2c. Feature engineering (matches train_model.py exactly) ---
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start=df.index.min(), end=df.index.max())
major_holidays = pd.to_datetime([
    "2024-01-01", "2025-01-01", "2026-01-01",
    "2024-11-28", "2025-11-27", "2026-11-26",
    "2024-12-25", "2025-12-25", "2026-12-25",
])

df["is_major_holiday"] = df.index.normalize().isin(major_holidays).astype(int)
df["is_minor_holiday"] = (
    df.index.normalize().isin(holidays) & ~df.index.normalize().isin(major_holidays)
).astype(int)

df["hour"]       = df.index.hour
df["dayofweek"]  = df.index.dayofweek
df["month"]      = df.index.month
df["weekofyear"] = df.index.isocalendar().week.astype(int)
df["is_january"] = (df["month"] == 1).astype(int)

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["day_sin"]  = np.sin(2 * np.pi * df["dayofweek"] / 7)
df["day_cos"]  = np.cos(2 * np.pi * df["dayofweek"] / 7)

tax_day = pd.Timestamp(f"{today.year}-04-15")
df["days_to_tax_day"] = (tax_day - today).days
df["is_tax_season"]   = int(today.month <= 4 and df["days_to_tax_day"].iloc[0] >= 0)
df["is_post_tax_drop"] = int(df["days_to_tax_day"].iloc[0] < 0 and df["days_to_tax_day"].iloc[0] > -31)

# Lag features — pull from the historical call_volume_history in the bundle
lag_col   = f"lag_{forecast_weeks}weeks"
trend_col = f"trend_{forecast_weeks}w"
max_col   = f"max_{forecast_weeks}w"

lag_offset = pd.Timedelta(weeks=forecast_weeks)
rolling_window = forecast_weeks * 7 * 48  # number of 30-min slots

# For each slot, look back N weeks in the saved history
lag_values = []
for ts in df.index:
    lag_ts = ts - lag_offset
    if lag_ts in call_volume_hist.index:
        lag_values.append(call_volume_hist.loc[lag_ts])
    else:
        # Find nearest available slot
        idx = call_volume_hist.index.get_indexer([lag_ts], method="nearest")
        lag_values.append(call_volume_hist.iloc[idx[0]])

df[lag_col] = lag_values

# Trend: rolling mean of call volume over the past N weeks ending at the lag point
# Max: rolling max over the same window
trend_values = []
max_values = []
for ts in df.index:
    lag_ts = ts - lag_offset
    window_start = lag_ts - pd.Timedelta(minutes=30 * rolling_window)
    window = call_volume_hist.loc[window_start:lag_ts]
    if len(window) > 0:
        trend_values.append(window.mean())
        max_values.append(window.max())
    else:
        trend_values.append(0.0)
        max_values.append(0.0)

df[trend_col] = trend_values
df[max_col]   = max_values

print("\nFeature matrix (first 5 slots):")
print(df[features].head())

Forecasting for: Friday, March 06, 2026

Feature matrix (first 5 slots):
                     hour_sin      hour_cos   day_sin   day_cos  month  \
2026-03-06 05:00:00  0.965926  2.588190e-01 -0.433884 -0.900969      3   
2026-03-06 05:30:00  0.965926  2.588190e-01 -0.433884 -0.900969      3   
2026-03-06 06:00:00  1.000000  6.123234e-17 -0.433884 -0.900969      3   
2026-03-06 06:30:00  1.000000  6.123234e-17 -0.433884 -0.900969      3   
2026-03-06 07:00:00  0.965926 -2.588190e-01 -0.433884 -0.900969      3   

                     weekofyear  is_january  is_minor_holiday  \
2026-03-06 05:00:00          10           0                 0   
2026-03-06 05:30:00          10           0                 0   
2026-03-06 06:00:00          10           0                 0   
2026-03-06 06:30:00          10           0                 0   
2026-03-06 07:00:00          10           0                 0   

                     is_major_holiday  days_to_tax_day  is_tax_season  \
2026-03-06 05:00:0

In [5]:
# --- 2d. Predict with RF + GB ensemble (average) ---
X = df[features]

pred_rf = rf_model.predict(X)
pred_gb = gb_model.predict(X)
pred_ensemble = np.maximum(0, np.round((pred_rf + pred_gb) / 2)).astype(int)

df["pred_rf"]       = np.round(pred_rf).astype(int)
df["pred_gb"]       = np.round(pred_gb).astype(int)
df["pred_ensemble"] = pred_ensemble

# Holiday override: if today is a major holiday, use historical holiday profile
if df["is_major_holiday"].iloc[0] == 1 and holiday_profiles:
    for ts in df.index:
        key = (ts.month, ts.day, ts.time())
        if key in holiday_profiles:
            df.loc[ts, "pred_ensemble"] = max(0, round(holiday_profiles[key]))
    print("Holiday override applied from historical profile.")

forecast_df = df[["pred_rf", "pred_gb", "pred_ensemble"]].copy()
forecast_df.index = forecast_df.index.strftime("%H:%M")
forecast_df.index.name = "Time"

print(f"\n{'='*55}")
print(f"  DEMAND FORECAST — {today.strftime('%A %B %d, %Y')}")
print(f"  Model: Random Forest + Gradient Boosting Ensemble")
print(f"{'='*55}")
print(forecast_df.to_string())
print(f"\nDaily total (ensemble): {pred_ensemble.sum():,} calls")
print(f"Peak slot: {forecast_df['pred_ensemble'].idxmax()} with {forecast_df['pred_ensemble'].max()} calls")


  DEMAND FORECAST — Friday March 06, 2026
  Model: Random Forest + Gradient Boosting Ensemble
       pred_rf  pred_gb  pred_ensemble
Time                                  
05:00      188       37            112
05:30      188       37            112
06:00      189       36            112
06:30      189       36            112
07:00      191       37            114
07:30      191       37            114
08:00      201       45            123
08:30      201       45            123
09:00      227       66            146
09:30      227       66            146
10:00      237       74            155
10:30      237       74            155
11:00      242       76            159
11:30      242       76            159
12:00      271      116            193
12:30      271      116            193
13:00      479      201            340
13:30      479      201            340
14:00     1506     1168           1337
14:30     1506     1168           1337
15:00     1778     1364           1571
15:30   

In [6]:
# =============================================================================
# Cell 3: The Erlang-A Optimizer (The Final Test)
# =============================================================================

AHT_SECONDS = 600  # Average Handle Time = 10 minutes

# Initialize the Erlang-A emulator
emulator = CallCenterEmulator(
    config=EmulatorConfig(
        avg_handle_time=AHT_SECONDS,
        sla_threshold_seconds=60,        # answer within 60 seconds
        interval_duration_seconds=1800,  # 30-min slots
        avg_patience_time=180,           # callers wait avg 3 min before hanging up
    ),
    model="erlang_a",
)

# Initialize the optimizer (binary search over agent counts)
optimizer = SupplyOptimizer(emulator, max_supply=500)

# Default constraints (these are what the UI sliders control)
constraints = OptimizationConstraints(
    min_sla=0.80,        # 80% SLA
    max_wait_time=60.0,  # max 60s avg wait
    max_occupancy=0.85,  # max 85% agent utilization
)

print(f"Emulator config:")
print(f"  AHT              : {AHT_SECONDS}s ({AHT_SECONDS // 60} min)")
print(f"  SLA threshold    : {emulator.config.sla_threshold_seconds}s")
print(f"  Interval         : {emulator.config.interval_duration_seconds}s (30 min)")
print(f"  Avg patience     : {emulator.config.avg_patience_time}s")
print(f"  Model            : Erlang-A")
print(f"\nConstraints:")
print(f"  Min SLA          : {constraints.min_sla * 100:.0f}%")
print(f"  Max wait time    : {constraints.max_wait_time}s")
print(f"  Max occupancy    : {constraints.max_occupancy * 100:.0f}%")

# --- Run the optimizer for each 30-min slot ---
results = []
for ts, row in df.iterrows():
    demand = int(row["pred_ensemble"])
    result = optimizer.optimize(
        demand=demand,
        constraints=constraints,
        avg_handle_time=AHT_SECONDS,
    )
    metrics = result.predicted_metrics
    results.append({
        "Time":            ts.strftime("%H:%M"),
        "Predicted Calls": demand,
        "AHT (s)":         AHT_SECONDS,
        "Required Agents": result.headcount,
        "Avg Wait (s)":    round(metrics.avg_wait_time, 1),
        "SLA %":           round(metrics.sla_compliance, 1),
        "Utilization %":   round(metrics.utilization_rate, 1),
        "Abandon %":       round(metrics.abandonment_rate, 1),
        "Feasible":        result.is_feasible,
    })

schedule = pd.DataFrame(results)

print(f"\n{'='*90}")
print(f"  STAFFING SCHEDULE — {today.strftime('%A %B %d, %Y')}")
print(f"  ML Forecast (RF+GB) → Erlang-A Optimizer")
print(f"{'='*90}")
print(schedule.to_string(index=False))
print(f"\n--- Summary ---")
print(f"Total predicted calls : {schedule['Predicted Calls'].sum():,}")
print(f"Peak agents needed   : {schedule['Required Agents'].max()} at {schedule.loc[schedule['Required Agents'].idxmax(), 'Time']}")
print(f"All slots feasible   : {schedule['Feasible'].all()}")

Emulator config:
  AHT              : 600s (10 min)
  SLA threshold    : 60s
  Interval         : 1800s (30 min)
  Avg patience     : 180s
  Model            : Erlang-A

Constraints:
  Min SLA          : 80%
  Max wait time    : 60.0s
  Max occupancy    : 85%

  STAFFING SCHEDULE — Friday March 06, 2026
  ML Forecast (RF+GB) → Erlang-A Optimizer
 Time  Predicted Calls  AHT (s)  Required Agents  Avg Wait (s)  SLA %  Utilization %  Abandon %  Feasible
05:00              112      600               46           5.9   85.3           81.2       11.1      True
05:30              112      600               46           5.9   85.3           81.2       11.1      True
06:00              112      600               46           5.9   85.3           81.2       11.1      True
06:30              112      600               46           5.9   85.3           81.2       11.1      True
07:00              114      600               46           7.8   81.4           82.6       13.9      True
07:30           